# Module 07 — Notebook 3: Seeds and Reproducibility

## Learning Objectives

By the end of this notebook, you will be able to:

- Explain what randomness means in code and why it breaks reproducibility
- Set seeds with `random.seed()` and `numpy.random.seed()`
- Use `numpy.random.default_rng()` for modern, self-contained seeding
- Write a standard seed-setting preamble for any analysis script
- Understand the limits of seed-based reproducibility

**Estimated time:** ~20 minutes

## Why This Matters for AI Research Engineering

Every ML paper's results depend on randomness: train/test splits, weight initialization, data shuffling, dropout masks, and sampling from model outputs. Without seeding, two researchers with identical code can get different results — a major source of "irreproducible" papers.

Setting seeds is the simplest thing you can do to make your work trustworthy.

In [ ]:
import sys
sys.path.insert(0, "../../")
from src.checks import check_equal, check_type, check_approx, check_contains, check_length
import random
import numpy as np
from pathlib import Path
print("Setup complete.")

## 1. The Problem: Randomness Changes Every Run

Without a seed, Python's random number generator starts from a different state each time — seeded by the system clock or OS entropy.

In [ ]:
# No seed — run this cell twice and you'll get different numbers each time
import random
sample = [round(random.random(), 4) for _ in range(5)]
print("Random sample:", sample)
print("Run this cell again — you'll get different numbers!")

## 2. Fixing It: random.seed()

Calling `random.seed(n)` before drawing numbers guarantees you get the same sequence every time you run the code with that seed.

In [ ]:
import random

# First run with seed 42
random.seed(42)
run1 = [round(random.random(), 4) for _ in range(5)]

# Reset the same seed
random.seed(42)
run2 = [round(random.random(), 4) for _ in range(5)]

print("Run 1:", run1)
print("Run 2:", run2)
print("Identical:", run1 == run2)  # True

In [ ]:
# The seed affects ALL random operations from the stdlib random module:
random.seed(42)
print("randint:", random.randint(1, 100))
print("choice: ", random.choice(["alpha", "beta", "gamma"]))

items = [1, 2, 3, 4, 5]
random.shuffle(items)
print("shuffle:", items)

print("sample: ", random.sample(range(100), 3))

## 3. NumPy Seeds

NumPy has its own random number generator, separate from Python's stdlib `random`. You need to seed it independently.

In [ ]:
import numpy as np

# Legacy interface — sets a global module-level seed
np.random.seed(42)
arr1 = np.random.randn(5).round(4)

np.random.seed(42)
arr2 = np.random.randn(5).round(4)

print("Legacy seed run 1:", arr1)
print("Legacy seed run 2:", arr2)
print("Equal:", np.array_equal(arr1, arr2))

In [ ]:
# Modern Generator API — preferred for new code (NumPy >= 1.17)
# The Generator is self-contained: two RNGs with different seeds won't interfere
rng = np.random.default_rng(seed=42)
arr3 = rng.standard_normal(5).round(4)

rng2 = np.random.default_rng(seed=42)
arr4 = rng2.standard_normal(5).round(4)

print("Generator run 1:", arr3)
print("Generator run 2:", arr4)
print("Equal:", np.array_equal(arr3, arr4))

## 4. The Standard Seed Preamble

Copy this pattern to the **top of every analysis script** — before any stochastic code runs.

In [ ]:
# Standard reproducibility preamble
import random
import numpy as np

SEED = 42  # name it so you can change it in one place

random.seed(SEED)
np.random.seed(SEED)
# If using PyTorch: torch.manual_seed(SEED)
# If using TensorFlow: tf.random.set_seed(SEED)

print(f"Seeds set. SEED={SEED}")

## 5. What Seeds Do NOT Fix

Seeds control Python and NumPy randomness, but some sources of non-determinism are outside their reach:

| Source | Controlled by seed? | Notes |
|---|---|---|
| `random.random()` | ✓ Yes | stdlib seed covers this |
| `np.random.randn()` | ✓ Yes | NumPy seed covers this |
| GPU operations (CUDA) | ✗ No | Floating-point order-of-ops varies on GPU |
| LLM API calls | ✗ No | Your Python seed doesn't reach the model server |
| Multiprocessing | ✗ Partial | Worker processes need their own seeds |
| File system ordering | ✗ No | `os.listdir()` order isn't guaranteed |

For GPU determinism in PyTorch:
```python
import torch
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
```
Note: this can slow down training significantly.

## Exercise 1 — Verify Seed Reproducibility

Set `random.seed(99)`, draw 3 random integers between 1 and 100 using `random.randint(1, 100)`, and store them in `draw_1` as a list. Then reset `random.seed(99)` and draw 3 more into `draw_2`. They should be identical.

In [ ]:
import random

# YOUR CODE HERE
draw_1 = None  # list of 3 ints
draw_2 = None  # list of 3 ints

In [ ]:
check_type(draw_1, list, "draw_1 is a list")
check_length(draw_1, 3, "draw_1 has 3 elements")
check_equal(draw_1, draw_2, "draw_1 and draw_2 are identical (seed reproducibility confirmed)")

## Exercise 2 — Seeded NumPy Mean

Set `np.random.seed(7)`, draw 100 samples from a standard normal distribution using `np.random.randn(100)`, compute the mean, round to 4 decimal places, and store in `sample_mean`.

In [ ]:
import numpy as np

# YOUR CODE HERE
sample_mean = None  # float, rounded to 4 decimal places

In [ ]:
check_type(sample_mean, float, "sample_mean is a float")
check_approx(sample_mean, 0.0121, 0.001, "sample_mean matches expected value for seed=7, n=100")

## Exercise 3 — Generator API

Create a `numpy` Generator with `np.random.default_rng(seed=123)`. Draw 50 samples from a uniform distribution (`rng.uniform(low=0, high=1, size=50)`). Store the maximum value, rounded to 4 decimal places, in `sample_max`.

In [ ]:
import numpy as np

# YOUR CODE HERE
sample_max = None  # float, rounded to 4 decimal places

In [ ]:
check_type(sample_max, float, "sample_max is a float")
check_approx(sample_max, 0.9274, 0.001, "sample_max matches expected value for seed=123, n=50")

## Exercise 4 — Write the Seed Preamble

Write a Python script `seed_preamble.py` that:
1. Imports `random` and `numpy as np`
2. Defines `SEED = 42`
3. Calls `random.seed(SEED)` and `np.random.seed(SEED)`
4. Prints `"Seeds set. SEED=42"`

In [ ]:
%%writefile seed_preamble.py
# YOUR CODE HERE
# Replace this comment with the actual preamble script
print("replace me")

In [ ]:
source = Path("seed_preamble.py").read_text()
check_contains(source, "SEED = 42", "SEED constant is defined")
check_contains(source, "random.seed", "random.seed is called")
check_contains(source, "np.random.seed", "np.random.seed is called")

## Wrap-Up

| Code | What it does |
|---|---|
| `random.seed(42)` | Seed Python's stdlib RNG |
| `np.random.seed(42)` | Seed NumPy's legacy global RNG |
| `rng = np.random.default_rng(42)` | Create a self-contained seeded Generator (preferred) |
| `SEED = 42` at module top | Make seed a named constant — easy to change in one place |
| `torch.manual_seed(42)` | Seed PyTorch (when applicable) |

**Remember:** Seeds control Python/NumPy randomness but not GPU operations, LLM API calls, or multiprocessing workers.

**Next:** Notebook 4 — Mini-Project: build a fully reproducible analysis setup